In [18]:
# =============================================================================
# NOTE: Copy each section (between the ╔═╗ banners) into its own Jupyter cell.
# =============================================================================
#
#  INTERCONNECT — 14-Ring Add-Drop Cascade · neff / ng Parametric Sweep
#  ──────────────────────────────────────────────────────────────────────
#  Ring model : UNIDIRECTIONAL (bidirectional = 0)
#    → Only forward-propagating modes are modelled.
#    → No backscattering / counter-propagating field.
#    → Faster convergence; appropriate for high-isolation cascades.
#
#  Circuit topology
#  ─────────────────
#  ONA output     ──► RING_1  port 1          (bus 1 input)
#  RING_1  port 3 ──► ONA     input 1         (through → ONA port 1)
#  RING_n  port 4 ──► RING_{n+1}  port 1      (drop feeds next ring, n=1..13)
#  RING_14 port 3 ──► ONA     input 2         (final through → ONA port 2)
#  RING_n  port 2 ──► OPM_{n-1}               (add port used as drop monitor)
#
#  Sweep axis
#  ──────────
#  Paired (neff_TE, ng_TE) for RING_1 only.
#  Rings 2-14 are held at their fixed parameter values.
#
#  v202 API notes
#  ──────────────
#  All circuit operations go through ic.eval("lsf;").
#  ic.getresult() IS a valid direct Python call.
#  Property names are resolved at runtime via candidate lists and cached.
# =============================================================================


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Imports · lumapi setup · Logging · I/O paths                     ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import sys
import os
import platform
import time
import logging
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass, asdict
from typing import List, Optional, Tuple, Dict, Any

import numpy as np
import h5py
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

# ─────────────────────────────────────────────────────────────────────────────
# Lumerical installation paths
# ─────────────────────────────────────────────────────────────────────────────
LUMERICAL_VERSION = "v202"

if platform.system() == "Windows":
    LUMERICAL_ROOT = rf"C:\Program Files\Lumerical\{LUMERICAL_VERSION}"
    LUMERICAL_API  = rf"{LUMERICAL_ROOT}\api\python"
    LUMERICAL_BIN  = rf"{LUMERICAL_ROOT}\bin"
else:
    LUMERICAL_ROOT = f"/opt/lumerical/{LUMERICAL_VERSION}"
    LUMERICAL_API  = f"{LUMERICAL_ROOT}/api/python"
    LUMERICAL_BIN  = f"{LUMERICAL_ROOT}/bin"

if "lumapi" in sys.modules:
    del sys.modules["lumapi"]

if LUMERICAL_API not in sys.path:
    sys.path.insert(0, LUMERICAL_API)

if platform.system() == "Windows":
    if hasattr(os, "add_dll_directory"):
        os.add_dll_directory(str(LUMERICAL_BIN))
    else:
        os.environ["PATH"] = str(LUMERICAL_BIN) + ";" + os.environ.get("PATH", "")

assert Path(LUMERICAL_API).exists(), (
    f"Lumerical API path not found:\n  {LUMERICAL_API}\n"
    f"Check LUMERICAL_VERSION = '{LUMERICAL_VERSION}'"
)
assert Path(LUMERICAL_BIN).exists(), \
    f"Lumerical bin path not found:\n  {LUMERICAL_BIN}"

import lumapi  # noqa — must come after all path setup
print(f"lumapi imported successfully from:\n  {lumapi.__file__}")

# ─────────────────────────────────────────────────────────────────────────────
# Logging
# ─────────────────────────────────────────────────────────────────────────────
logging.basicConfig(
    level   = logging.INFO,
    format  = "%(asctime)s │ %(levelname)s │ %(message)s",
    datefmt = "%H:%M:%S",
)
log = logging.getLogger("ICNT_Cascade")

# ─────────────────────────────────────────────────────────────────────────────
# I/O paths
# ─────────────────────────────────────────────────────────────────────────────
VERSION_NAME = "ICNT_14Ring_Cascade_UniDir_neff_sweep_V1"
PROJECT_DIR  = Path.cwd()
DATA_DIR     = PROJECT_DIR / "data_ICNT_cascade_ring_sweep"
DATA_DIR.mkdir(parents=True, exist_ok=True)
HDF5_PATH    = DATA_DIR / f"{VERSION_NAME}.h5"
FIGURES_DIR  = DATA_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n  Data directory : {DATA_DIR}")
print(f"  HDF5 output    : {HDF5_PATH}")
print(f"  Figures dir    : {FIGURES_DIR}")

# ═════════════════════════════════════════════════════════════════════════════
#  END CELL 1
# ═════════════════════════════════════════════════════════════════════════════


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — All simulation parameters defined as arrays                      ║
# ║                                                                            ║
# ║  ► Edit the values in this cell only.                                      ║
# ║  ► One entry per ring (index 0 = Ring 1, index 13 = Ring 14).              ║
# ║  ► All length / wavelength quantities in SI metres unless noted.           ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

N_RINGS = 14

# ─────────────────────────────────────────────────────────────────────────────
# Ring radius [m]
# ─────────────────────────────────────────────────────────────────────────────
RING_RADIUS_M = np.array([
    19.0021e-6,   # Ring  1
    19.1818e-6,   # Ring  2
    19.1934e-6,   # Ring  3
    19.2051e-6,   # Ring  4
    19.2168e-6,   # Ring  5
    19.2284e-6,   # Ring  6
    19.2401e-6,   # Ring  7
    19.2517e-6,   # Ring  8
    19.4158e-6,   # Ring  9
    19.4275e-6,   # Ring 10
    19.4393e-6,   # Ring 11
    19.4511e-6,   # Ring 12
    19.4628e-6,   # Ring 13
    19.4746e-6,   # Ring 14
])

# ─────────────────────────────────────────────────────────────────────────────
# Resonance / expansion wavelength [m]
# ─────────────────────────────────────────────────────────────────────────────
RING_LAMBDA_RES_M = np.array([
    1.5500000e-6,   # Ring  1
    1.5500000e-6,   # Ring  2
    1.5507692e-6,   # Ring  3
    1.5515385e-6,   # Ring  4
    1.5523077e-6,   # Ring  5
    1.5530769e-6,   # Ring  6
    1.5538462e-6,   # Ring  7
    1.5546154e-6,   # Ring  8
    1.5553846e-6,   # Ring  9
    1.5561538e-6,   # Ring 10
    1.5569231e-6,   # Ring 11
    1.5576923e-6,   # Ring 12
    1.5584615e-6,   # Ring 13
    1.5582308e-6,   # Ring 14
])

# ─────────────────────────────────────────────────────────────────────────────
# TE effective index  (Ring 1 = sweep default; swept in Cell 3)
# ─────────────────────────────────────────────────────────────────────────────
RING_NEFF_TE = np.array([
    1.609803,   # Ring  1  ← swept
    1.633303,   # Ring  2
    1.633121,   # Ring  3
    1.632939,   # Ring  4
    1.632758,   # Ring  5
    1.632576,   # Ring  6
    1.632394,   # Ring  7
    1.632213,   # Ring  8
    1.631974,   # Ring  9
    1.631792,   # Ring 10
    1.631611,   # Ring 11
    1.631430,   # Ring 12
    1.631248,   # Ring 13
    1.631067,   # Ring 14
])

# ─────────────────────────────────────────────────────────────────────────────
# TE group index  (Ring 1 = sweep default; swept in Cell 3)
# ─────────────────────────────────────────────────────────────────────────────
RING_NG_TE = np.array([
    2.020543,   # Ring  1  ← swept
    1.991101,   # Ring  2
    1.990956,   # Ring  3
    1.990808,   # Ring  4
    1.990659,   # Ring  5
    1.990509,   # Ring  6
    1.990356,   # Ring  7
    1.990203,   # Ring  8
    1.990047,   # Ring  9
    1.989891,   # Ring 10
    1.989733,   # Ring 11
    1.989691,   # Ring 12
    1.989528,   # Ring 13
    1.989364,   # Ring 14
])

# ─────────────────────────────────────────────────────────────────────────────
# GVD dispersion [ps²/km]  →  converted internally to s²/m (× 1e-15)
# ─────────────────────────────────────────────────────────────────────────────
RING_D_TE_PS2_PER_KM = np.zeros(N_RINGS)
RING_D_TM_PS2_PER_KM = np.zeros(N_RINGS)

# ─────────────────────────────────────────────────────────────────────────────
# TM mode parameters  (used only when RING_POLARIZATION[i] == "TM")
# ─────────────────────────────────────────────────────────────────────────────
RING_NEFF_TM = np.full(N_RINGS, 1.7000)
RING_NG_TM   = np.full(N_RINGS, 2.1000)

# ─────────────────────────────────────────────────────────────────────────────
# Input coupler power coupling coefficient  |κ|²
# ─────────────────────────────────────────────────────────────────────────────
RING_KAPPA_INPUT_SQ = np.array([
    0.145778,   # Ring  1
    0.145072,   # Ring  2
    0.145011,   # Ring  3
    0.144949,   # Ring  4
    0.144888,   # Ring  5
    0.144827,   # Ring  6
    0.144765,   # Ring  7
    0.144704,   # Ring  8
    0.145696,   # Ring  9
    0.145634,   # Ring 10
    0.145572,   # Ring 11
    0.145518,   # Ring 12
    0.145456,   # Ring 13
    0.145394,   # Ring 14
])

# ─────────────────────────────────────────────────────────────────────────────
# Drop coupler power coupling coefficient  |κ|²
# ─────────────────────────────────────────────────────────────────────────────
RING_KAPPA_DROP_SQ = np.array([
    0.143402,   # Ring  1
    0.142672,   # Ring  2
    0.142609,   # Ring  3
    0.142546,   # Ring  4
    0.142484,   # Ring  5
    0.142420,   # Ring  6
    0.142357,   # Ring  7
    0.142294,   # Ring  8
    0.143269,   # Ring  9
    0.143205,   # Ring 10
    0.143142,   # Ring 11
    0.143086,   # Ring 12
    0.143022,   # Ring 13
    0.142958,   # Ring 14
])

# ─────────────────────────────────────────────────────────────────────────────
# Propagation loss [dB/m]
# ─────────────────────────────────────────────────────────────────────────────
RING_LOSS_DB_PER_M = np.full(N_RINGS, 101.0)

# ─────────────────────────────────────────────────────────────────────────────
# Active polarization: "TE" or "TM" per ring
# ─────────────────────────────────────────────────────────────────────────────
RING_POLARIZATION = ["TE"] * N_RINGS

# ─────────────────────────────────────────────────────────────────────────────
# ONA parameters
# ─────────────────────────────────────────────────────────────────────────────
ONA_LAMBDA_START_M  = 1.540e-6
ONA_LAMBDA_STOP_M   = 1.560e-6
ONA_N_POINTS        = 1000
ONA_POWER_DBM       = 0.0
ONA_N_INPUT_PORTS   = 2

# ─────────────────────────────────────────────────────────────────────────────
# Sweep parameters — paired (neff_TE, ng_TE) for RING_1 only
# Both arrays must have the same length = SWEEP_N_POINTS.
# ─────────────────────────────────────────────────────────────────────────────
SWEEP_N_POINTS = 21

SWEEP_NEFF = np.linspace(1.90, 2.10, SWEEP_N_POINTS)
SWEEP_NG   = np.linspace(2.20, 2.45, SWEEP_N_POINTS)

# Pattern B — uncomment for irregular steps:
# SWEEP_NEFF = np.array([1.90, 1.92, 1.95, 1.98, 2.00, 2.02, 2.05, 2.08, 2.10])
# SWEEP_NG   = np.array([2.20, 2.22, 2.25, 2.28, 2.30, 2.32, 2.35, 2.39, 2.45])
# SWEEP_N_POINTS = len(SWEEP_NEFF)

# ─────────────────────────────────────────────────────────────────────────────
# Validation
# ─────────────────────────────────────────────────────────────────────────────
assert len(RING_RADIUS_M)       == N_RINGS
assert len(RING_LAMBDA_RES_M)   == N_RINGS
assert len(RING_NEFF_TE)        == N_RINGS
assert len(RING_NG_TE)          == N_RINGS
assert len(RING_KAPPA_INPUT_SQ) == N_RINGS
assert len(RING_KAPPA_DROP_SQ)  == N_RINGS
assert len(RING_LOSS_DB_PER_M)  == N_RINGS
assert len(RING_POLARIZATION)   == N_RINGS
assert len(SWEEP_NEFF) == SWEEP_N_POINTS
assert len(SWEEP_NG)   == SWEEP_N_POINTS
assert len(SWEEP_NEFF) == len(SWEEP_NG)

print("=" * 72)
print("  INTERCONNECT 14-Ring Cascade — Parameter Summary  [UNIDIRECTIONAL]")
print("=" * 72)
print(f"  {'Ring':>5}  {'R [µm]':>9}  {'λ_res [nm]':>12}  "
      f"{'neff_TE':>9}  {'ng_TE':>8}  {'κ²_in':>8}  {'κ²_dr':>8}  {'Loss':>7}")
print("  " + "─" * 68)
for i in range(N_RINGS):
    tag = "  ← swept" if i == 0 else ""
    print(f"  {i+1:>5}  {RING_RADIUS_M[i]*1e6:>9.4f}  "
          f"{RING_LAMBDA_RES_M[i]*1e9:>12.4f}  "
          f"{RING_NEFF_TE[i]:>9.6f}  {RING_NG_TE[i]:>8.6f}  "
          f"{RING_KAPPA_INPUT_SQ[i]:>8.6f}  {RING_KAPPA_DROP_SQ[i]:>8.6f}  "
          f"{RING_LOSS_DB_PER_M[i]:>7.1f}{tag}")
print()
print(f"  ONA   : λ {ONA_LAMBDA_START_M*1e9:.2f}–{ONA_LAMBDA_STOP_M*1e9:.2f} nm  "
      f"│  {ONA_N_POINTS} pts  │  {ONA_POWER_DBM} dBm")
print(f"  Sweep : neff {SWEEP_NEFF[0]:.5f}→{SWEEP_NEFF[-1]:.5f}  "
      f"│  ng {SWEEP_NG[0]:.5f}→{SWEEP_NG[-1]:.5f}  │  {SWEEP_N_POINTS} pts")
print(f"  Ring model : UNIDIRECTIONAL  (bidirectional = 0)")
print("=" * 72)

# ═════════════════════════════════════════════════════════════════════════════
#  END CELL 2
# ═════════════════════════════════════════════════════════════════════════════


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Circuit builder + sweep engine + HDF5 storage                    ║
# ║                                                                            ║
# ║  UNIDIRECTIONAL CHANGE vs reference document:                              ║
# ║  • RING_PROP["bidir"] = ["bidirectional"] — set to 0 on every ring.       ║
# ║  • _apply_ring_params calls _set_first_valid(..., 0, "bidirectional").     ║
# ║  • All other topology, connections, HDF5, and plotting: unchanged.        ║
# ║                                                                            ║
# ║  Other architecture notes (inherited from reference document):             ║
# ║  • Every command in its own _eval() — no monolithic blocks.               ║
# ║  • Property names resolved at runtime, cached in _RESOLVED dict.          ║
# ║  • ICScriptError shows the exact failed command.                           ║
# ║  • c = 299_792_458 m/s exact (removes 0.07% frequency axis bias).        ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

SPEED_OF_LIGHT = 299_792_458.0    # m/s — exact SI definition

ONA_NAME = "ONA_1"
def ring_name(ring_id: int) -> str: return f"RING_{ring_id}"
def opm_name(opm_id:  int) -> str:  return f"OPM_{opm_id}"

# ─────────────────────────────────────────────────────────────────────────────
# Contextual error class
# ─────────────────────────────────────────────────────────────────────────────
class ICScriptError(RuntimeError):
    """INTERCONNECT scripting error with the exact failed command embedded."""

# ─────────────────────────────────────────────────────────────────────────────
# Scripting primitives
# ─────────────────────────────────────────────────────────────────────────────
def _eval(ic, cmd: str) -> None:
    """Execute ONE LSF command. Raises ICScriptError showing what failed."""
    cmd = cmd.strip()
    if not cmd.endswith(";"):
        cmd += ";"
    try:
        ic.eval(cmd)
    except Exception as exc:
        raise ICScriptError(
            f"\n  INTERCONNECT rejected command:\n"
            f"    {cmd}\n"
            f"  Original error: {exc}"
        ) from exc

def _try_eval(ic, cmd: str) -> bool:
    """Try ONE command. Returns True/False; never raises. Used for probing."""
    cmd = cmd.strip()
    if not cmd.endswith(";"):
        cmd += ";"
    try:
        ic.eval(cmd)
        return True
    except Exception:
        return False

def _fmt(value) -> str:
    """Convert a Python value to its LSF numeric/string literal."""
    if isinstance(value, str):
        return f'"{value}"'
    if isinstance(value, (bool, np.bool_)):
        return "1" if value else "0"
    if isinstance(value, (int, np.integer)):
        return str(int(value))
    return f"{float(value):.12e}"

# ─────────────────────────────────────────────────────────────────────────────
# Add elements with library-name candidates
# ─────────────────────────────────────────────────────────────────────────────
def _add_element(ic, library_candidates: List[str], name: str,
                 x_um: float, y_um: float) -> str:
    """
    Add an element trying each library name in order.
    Sets name and layout position on the selected element.
    Returns the library name that worked.
    """
    for lib in library_candidates:
        if _try_eval(ic, f'addelement("{lib}")'):
            _eval(ic, f'set("name", "{name}")')
            _try_eval(ic, f'set("x position", {x_um})')
            _try_eval(ic, f'set("y position", {y_um})')
            log.debug(f"  addelement OK: '{lib}' → '{name}'")
            return lib
    raise ICScriptError(
        f"\n  Could not add '{name}'.\n"
        f"  None of these library names exist in this INTERCONNECT version:\n"
        f"    {library_candidates}\n"
        f"  ► Open Element Library in GUI, find the element and copy the\n"
        f"    EXACT name (case-sensitive)."
    )

# ─────────────────────────────────────────────────────────────────────────────
# setnamed with candidate list — resolves and caches the correct property name
# ─────────────────────────────────────────────────────────────────────────────
def _set_first_valid(ic, element_name: str, cache_key: str,
                     prop_candidates: List[str], value,
                     label: str, required: bool = False) -> Optional[str]:
    """
    Try setnamed with each candidate property name.
    Caches the first accepted name in _RESOLVED[cache_key].
    If required=True and nothing works, raises ICScriptError.
    """
    lit = _fmt(value)

    # Fast path: name already resolved in a previous call
    cached = _RESOLVED.get(cache_key)
    if cached is not None:
        if _try_eval(ic, f'setnamed("{element_name}", "{cached}", {lit})'):
            return cached
        log.debug(f"  Cached '{cached}' failed for '{element_name}'; re-probing.")

    for prop in prop_candidates:
        if _try_eval(ic, f'setnamed("{element_name}", "{prop}", {lit})'):
            _RESOLVED[cache_key] = prop
            log.debug(f"  Resolved: '{cache_key}' = '{prop}' on '{element_name}'")
            return prop

    msg = (
        f"  ['{element_name}'] No valid property found for '{label}'.\n"
        f"  Candidates tried: {prop_candidates}\n"
        f"  ► Check exact name in GUI (double-click → Properties)."
    )
    if required:
        raise ICScriptError("\n" + msg + "\n  This property is required.")
    log.warning(msg + " — element default will be used.")
    return None

# ─────────────────────────────────────────────────────────────────────────────
# Library name candidates
# Add more entries if your version uses a different name.
# ─────────────────────────────────────────────────────────────────────────────
RING_LIBRARY_CANDIDATES = [
    "Double Bus Ring Resonator",
]

ONA_LIBRARY_CANDIDATES = [
    "Optical Network Analyzer",
]

OPM_LIBRARY_CANDIDATES = [
    "Optical Power Meter",
]

# ─────────────────────────────────────────────────────────────────────────────
# Property name candidates per element type
# ─────────────────────────────────────────────────────────────────────────────
ONA_PROP: Dict[str, List[str]] = {
    "n_input_ports": ["number of input ports"],
    "input_param":   ["input parameter"],
    "start_freq":    ["start frequency"],
    "stop_freq":     ["stop frequency"],
    "n_points":      ["number of points"],
    "power":         ["power"],
}

RING_PROP: Dict[str, List[str]] = {
    # ── Physics parameters ────────────────────────────────────────────────────
    "radius":  ["length"],
    "cwl":     ["frequency"],
    "neff":    ["effective index 1"],
    "ng":      ["group index 1"],
    "loss":    ["loss 1"],
    "disp":    ["dispersion 1"],
    "kappa1":  ["coupling coefficient 1 1"],
    "kappa2":  ["coupling coefficient 1 2"],
    # ── Directionality ───────────────────────────────────────────────────────
    # Setting this to 0 switches the model from bidirectional (default) to
    # unidirectional — no backward-propagating field is computed.
    # The property name varies across INTERCONNECT versions; all known
    # candidates are listed below. At least one must succeed.
    "bidir":   ["configuration"],
}

# Runtime cache of resolved property names (built once during _build_circuit,
# reused across all subsequent sweep-loop calls → zero repeated probing)
_RESOLVED: Dict[str, Optional[str]] = {}

# ─────────────────────────────────────────────────────────────────────────────
# Apply all parameters for one ring
# ─────────────────────────────────────────────────────────────────────────────
def _apply_ring_params(ic, ring_idx: int,
                       neff_override: Optional[float] = None,
                       ng_override:   Optional[float] = None) -> None:
    """
    Set all physics + directionality parameters for the ring at ring_idx.

    UNIDIRECTIONAL CHANGE
    ─────────────────────
    The last _set_first_valid call sets  bidirectional = 0.
    This disables the counter-propagating mode inside the ring:
      • No reflected field at the input port.
      • No mode splitting (no doublet resonances from backscattering).
      • Faster INTERCONNECT solver convergence.
      • Appropriate for high-isolation cascaded WDM / biosensor designs
        where backward coupling between channels is negligible.

    All other parameters and the call signature are identical to the
    bidirectional version — only one extra property is set.
    """
    name = ring_name(ring_idx + 1)
    neff = neff_override if neff_override is not None else float(RING_NEFF_TE[ring_idx])
    ng   = ng_override   if ng_override   is not None else float(RING_NG_TE[ring_idx])
    pol  = RING_POLARIZATION[ring_idx].upper()
    d_si = (float(RING_D_TE_PS2_PER_KM[ring_idx] if pol == "TE"
                  else RING_D_TM_PS2_PER_KM[ring_idx])
            * 1e-15)   # ps²/km → s²/m

    # ── Physics parameters ────────────────────────────────────────────────────
    _set_first_valid(ic, name, "ring_radius", RING_PROP["radius"],
                     RING_RADIUS_M[ring_idx],      "radius")
    _set_first_valid(ic, name, "ring_cwl",    RING_PROP["cwl"],
                     RING_LAMBDA_RES_M[ring_idx],  "center wavelength")
    _set_first_valid(ic, name, "ring_neff",   RING_PROP["neff"],
                     neff,                         "effective index",  required=True)
    _set_first_valid(ic, name, "ring_ng",     RING_PROP["ng"],
                     ng,                           "group index",      required=True)
    _set_first_valid(ic, name, "ring_loss",   RING_PROP["loss"],
                     RING_LOSS_DB_PER_M[ring_idx], "loss")
    _set_first_valid(ic, name, "ring_disp",   RING_PROP["disp"],
                     d_si,                         "dispersion")
    _set_first_valid(ic, name, "ring_kappa1", RING_PROP["kappa1"],
                     RING_KAPPA_INPUT_SQ[ring_idx],"coupling 1")
    _set_first_valid(ic, name, "ring_kappa2", RING_PROP["kappa2"],
                     RING_KAPPA_DROP_SQ[ring_idx], "coupling 2")

    # ── Unidirectional flag ───────────────────────────────────────────────────
    # Value 0 = unidirectional (forward mode only).
    # Value 1 = bidirectional  (forward + backward modes, default).
    # "required=False" — if the property doesn't exist in this version,
    # the element runs in its default mode rather than crashing the sweep.
    _set_first_valid(ic, name, "ring_bidir",  RING_PROP["bidir"],
                     0,                            "bidirectional",   required=False)

# ─────────────────────────────────────────────────────────────────────────────
# Lightweight Ring 1 update for the sweep loop (neff/ng only — no re-probing)
# ─────────────────────────────────────────────────────────────────────────────
def _update_ring1_neff_ng(ic, neff: float, ng: float) -> None:
    """
    Update only neff and ng of Ring 1 using cached property names.
    Called once per sweep step — must be as fast as possible.
    Falls back to full _apply_ring_params if cache is missing
    (should never happen after a successful _build_circuit).
    """
    name = ring_name(1)
    pn   = _RESOLVED.get("ring_neff")
    pg   = _RESOLVED.get("ring_ng")
    if pn and pg:
        _eval(ic, f'setnamed("{name}", "{pn}", {neff:.12f})')
        _eval(ic, f'setnamed("{name}", "{pg}", {ng:.12f})')
    else:
        # Fallback: full parameter set (only on first call if build was partial)
        _apply_ring_params(ic, 0, neff_override=neff, ng_override=ng)

# ─────────────────────────────────────────────────────────────────────────────
# Build the full circuit
# ─────────────────────────────────────────────────────────────────────────────
def _build_circuit(ic) -> None:
    """
    Build the 14-ring unidirectional cascade from scratch.

    Port map — Double Bus Ring Resonator (Ansys official):
      port 1 = bus 1 input      port 3 = bus 1 output (through)
      port 2 = bus 2 input      port 4 = bus 2 output (drop)

    OPMs on port 2 (bus 2 input / add port):
      Port 4 of rings 2-14 is used by the drop chain → next ring.
      Port 2 is free in passive unidirectional operation and carries
      the drop-channel power as a monitor output.
    """
    _eval(ic, "switchtodesign")
    _try_eval(ic, "selectall")
    _try_eval(ic, "delete")

    dx_um   = 200.0
    pwr_W   = 10.0 ** (ONA_POWER_DBM / 10.0) * 1e-3
    f_start = SPEED_OF_LIGHT / ONA_LAMBDA_STOP_M    # low f  = long  λ
    f_stop  = SPEED_OF_LIGHT / ONA_LAMBDA_START_M   # high f = short λ

    # ── ONA ───────────────────────────────────────────────────────────────────
    _add_element(ic, ONA_LIBRARY_CANDIDATES, ONA_NAME, 0.0, 0.0)
    _set_first_valid(ic, ONA_NAME, "ona_input_param", ONA_PROP["input_param"],
                     "start and stop", "input parameter")
    _set_first_valid(ic, ONA_NAME, "ona_n_ports",     ONA_PROP["n_input_ports"],
                     int(ONA_N_INPUT_PORTS),  "number of input ports")
    _set_first_valid(ic, ONA_NAME, "ona_start_freq",  ONA_PROP["start_freq"],
                     float(f_start), "start frequency", required=True)
    _set_first_valid(ic, ONA_NAME, "ona_stop_freq",   ONA_PROP["stop_freq"],
                     float(f_stop),  "stop frequency",  required=True)
    _set_first_valid(ic, ONA_NAME, "ona_n_points",    ONA_PROP["n_points"],
                     int(ONA_N_POINTS), "number of points")
    _set_first_valid(ic, ONA_NAME, "ona_power",       ONA_PROP["power"],
                     float(pwr_W),  "input power")
    log.info("  ONA added and configured.")

    # ── Rings (unidirectional) ────────────────────────────────────────────────
    for i in range(N_RINGS):
        _add_element(ic, RING_LIBRARY_CANDIDATES, ring_name(i + 1),
                     (i + 1) * dx_um, 0.0)
        _apply_ring_params(ic, ring_idx=i)
        log.info(f"  Ring {i+1} added  [unidir, neff={RING_NEFF_TE[i]:.6f}].")

    # ── OPMs ──────────────────────────────────────────────────────────────────
    n_opm = N_RINGS - 1
    for k in range(1, n_opm + 1):
        _add_element(ic, OPM_LIBRARY_CANDIDATES, opm_name(k),
                     (k + 1) * dx_um, -300.0)
    log.info(f"  {n_opm} OPMs added.")

    # ── Connections ───────────────────────────────────────────────────────────
    def wire(a, pa, b, pb):
        _eval(ic, f'connect("{a}", "{pa}", "{b}", "{pb}")')

    wire(ONA_NAME,          "output",  ring_name(1),    "port 1")  # source → R1
    wire(ring_name(1),      "port 3",  ONA_NAME,         "input 1") # R1 through → ONA
    for i in range(1, N_RINGS):                                      # drop chain
        wire(ring_name(i),  "port 4",  ring_name(i+1),  "port 1")
    wire(ring_name(N_RINGS),"port 3",  ONA_NAME,         "input 2") # R14 through → ONA
    for i in range(2, N_RINGS + 1):                                  # drop monitors
        wire(ring_name(i),  "port 2",  opm_name(i-1),   "input")

    resolved_str = "  |  ".join(
        f"{k}='{v}'" for k, v in _RESOLVED.items() if v is not None
    )
    log.info(f"Circuit built: {N_RINGS} rings (UNIDIRECTIONAL), {n_opm} OPMs, 1 ONA.")
    log.info(f"Resolved props → {resolved_str}")

# ─────────────────────────────────────────────────────────────────────────────
# Extract results after a completed run
# ─────────────────────────────────────────────────────────────────────────────
def _extract_results(ic) -> Tuple[np.ndarray, np.ndarray,
                                   np.ndarray, np.ndarray, np.ndarray]:
    n_opm = N_RINGS - 1
    raw_1  = ic.getresult(ONA_NAME, "input 1/mode 1/transmission")
    f_arr  = np.asarray(raw_1["f"]).flatten()
    sort_i = np.argsort(f_arr)[::-1]              # ascending in wavelength
    wl_m   = SPEED_OF_LIGHT / f_arr[sort_i]

    def _T_dB(port_label: str) -> np.ndarray:
        raw = ic.getresult(ONA_NAME, f"{port_label}/mode 1/transmission")
        T   = np.asarray(raw["T"]).flatten()[sort_i]
        T   = np.where(T > 0, T, 1e-30)
        return 10.0 * np.log10(np.abs(T))

    T_port1_dB = _T_dB("input 1")
    T_port2_dB = _T_dB("input 2")

    n_wl           = len(wl_m)
    opm_power_dBm  = np.full(n_opm, np.nan)
    opm_spectrum_W = np.full((n_opm, n_wl), np.nan)

    for k in range(1, n_opm + 1):
        try:
            p_arr = np.asarray(ic.getresult(opm_name(k), "power")).flatten()
            if p_arr.size == n_wl:
                opm_spectrum_W[k-1, :] = (p_arr[sort_i]
                                          if p_arr.size == f_arr.size else p_arr)
            elif p_arr.size > 0:
                opm_spectrum_W[k-1, :] = p_arr[0]
            mean_p = np.nanmean(opm_spectrum_W[k-1, :])
            opm_power_dBm[k-1] = 10.0 * np.log10(mean_p * 1e3 + 1e-40)
        except Exception as exc:
            log.warning(f"  OPM {k} extraction failed: {exc}")

    return wl_m, T_port1_dB, T_port2_dB, opm_power_dBm, opm_spectrum_W

# ─────────────────────────────────────────────────────────────────────────────
# HDF5 initialisation
# ─────────────────────────────────────────────────────────────────────────────
def _init_hdf5(wl_ref_m: np.ndarray) -> None:
    n_pts, n_opm, n_wl = SWEEP_N_POINTS, N_RINGS - 1, len(wl_ref_m)
    with h5py.File(HDF5_PATH, "w") as f:
        md = f.create_group("metadata")
        md.create_dataset("neff_sweep",    data=SWEEP_NEFF)
        md.create_dataset("ng_sweep",      data=SWEEP_NG)
        md.create_dataset("wavelengths_m", data=wl_ref_m)
        md.attrs["version_name"]         = VERSION_NAME
        md.attrs["n_rings"]              = N_RINGS
        md.attrs["n_opm"]                = n_opm
        md.attrs["sweep_n_points"]       = SWEEP_N_POINTS
        md.attrs["ring_model"]           = "unidirectional"   # ← new tag
        md.attrs["ona_lambda_start_m"]   = ONA_LAMBDA_START_M
        md.attrs["ona_lambda_stop_m"]    = ONA_LAMBDA_STOP_M
        md.attrs["ona_n_points"]         = ONA_N_POINTS
        md.attrs["ona_power_dBm"]        = ONA_POWER_DBM
        md.attrs["timestamp_start"]      = datetime.now().isoformat()
        for i in range(N_RINGS):
            p = f"ring{i+1}_"
            md.attrs[p + "radius_m"]       = RING_RADIUS_M[i]
            md.attrs[p + "lambda_res_m"]   = RING_LAMBDA_RES_M[i]
            md.attrs[p + "neff_TE"]        = RING_NEFF_TE[i]
            md.attrs[p + "ng_TE"]          = RING_NG_TE[i]
            md.attrs[p + "kappa_input_sq"] = RING_KAPPA_INPUT_SQ[i]
            md.attrs[p + "kappa_drop_sq"]  = RING_KAPPA_DROP_SQ[i]
            md.attrs[p + "loss_dB_per_m"]  = RING_LOSS_DB_PER_M[i]
            md.attrs[p + "polarization"]   = RING_POLARIZATION[i]
            md.attrs[p + "bidirectional"]  = 0                # ← unidirectional flag
        rg = f.create_group("results")
        rg.create_dataset("T_port1_dB",
                          data=np.full((n_pts, n_wl), np.nan),
                          chunks=(1, n_wl))
        rg.create_dataset("T_port2_dB",
                          data=np.full((n_pts, n_wl), np.nan),
                          chunks=(1, n_wl))
        rg.create_dataset("opm_power_dBm",
                          data=np.full((n_pts, n_opm), np.nan),
                          chunks=(1, n_opm))
        rg.create_dataset("opm_spectrum_W",
                          data=np.full((n_pts, n_opm, n_wl), np.nan),
                          chunks=(1, n_opm, n_wl))
        f.create_group("flags").create_dataset(
            "computed", data=np.zeros(n_pts, dtype=bool), chunks=(1,))
    log.info(f"HDF5 initialised → {HDF5_PATH}")

# ─────────────────────────────────────────────────────────────────────────────
# Main sweep function
# ─────────────────────────────────────────────────────────────────────────────
def run_interconnect_sweep(hide_gui: bool = False) -> Dict[str, Any]:
    """
    Run the parametric (neff_TE, ng_TE) sweep for RING_1 (unidirectional cascade).
    - Rings 2-14 are held at their fixed parameter values.
    - Writes to HDF5 after every point (fault-safe, resumable).
    - Skips already-computed points on re-run.

    Returns dict: neff_sweep, ng_sweep, wavelengths_m,
                  T_port1_dB, T_port2_dB,
                  opm_power_dBm, opm_spectrum_W, computed
    """
    n_pts, n_opm = SWEEP_N_POINTS, N_RINGS - 1
    wavelengths_m = T_port1_dB = T_port2_dB = opm_power_dBm = opm_spectrum_W = None
    computed   = np.zeros(n_pts, dtype=bool)
    hdf5_ready = False

    # ── Cache check ───────────────────────────────────────────────────────────
    if HDF5_PATH.exists():
        log.info(f"Cache found → {HDF5_PATH}")
        with h5py.File(HDF5_PATH, "r") as f:
            wavelengths_m  = f["metadata/wavelengths_m"][:]
            T_port1_dB     = f["results/T_port1_dB"][:]
            T_port2_dB     = f["results/T_port2_dB"][:]
            opm_power_dBm  = f["results/opm_power_dBm"][:]
            opm_spectrum_W = f["results/opm_spectrum_W"][:]
            computed[:]    = f["flags/computed"][:]
        hdf5_ready = True
        n_cached   = int(computed.sum())
        remaining  = n_pts - n_cached
        log.info(f"Cached: {n_cached}/{n_pts}  |  Remaining: {remaining}")
        if remaining == 0:
            log.info("All sweep points cached — returning stored results.")
            return _pack_results(wavelengths_m, T_port1_dB, T_port2_dB,
                                 opm_power_dBm, opm_spectrum_W, computed)
    else:
        log.info("No cache found — starting fresh.")

    # ── Open INTERCONNECT ─────────────────────────────────────────────────────
    log.info("Launching Lumerical INTERCONNECT …")
    ic          = lumapi.INTERCONNECT(hide=hide_gui)
    runs_done   = 0
    runs_total  = int((~computed).sum())
    t_start     = time.time()

    try:
        _build_circuit(ic)
        log.info(f"Circuit ready — {runs_total} sweep point(s) remaining …")

        for s_idx in range(n_pts):
            if computed[s_idx]:
                continue

            neff_val = float(SWEEP_NEFF[s_idx])
            ng_val   = float(SWEEP_NG[s_idx])

            # Update Ring 1 neff / ng only — minimum work per step
            _eval(ic, "switchtodesign")
            _update_ring1_neff_ng(ic, neff_val, ng_val)

            # Run
            try:
                _eval(ic, "run")
            except ICScriptError as exc:
                log.warning(f"  RUN FAILED │ pt {s_idx} │ "
                             f"neff={neff_val:.6f} │ {exc}")
                computed[s_idx] = True
                if hdf5_ready:
                    with h5py.File(HDF5_PATH, "r+") as hf:
                        hf["flags/computed"][s_idx] = True
                        hf.flush()
                continue

            # Extract
            try:
                wl_m, t1, t2, opm_p, opm_s = _extract_results(ic)
            except Exception as exc:
                log.warning(f"  EXTRACT FAILED │ pt {s_idx} │ {exc}")
                computed[s_idx] = True
                continue

            # Initialise arrays + HDF5 on first successful run
            if wavelengths_m is None:
                n_wl           = len(wl_m)
                wavelengths_m  = wl_m
                T_port1_dB     = np.full((n_pts, n_wl), np.nan)
                T_port2_dB     = np.full((n_pts, n_wl), np.nan)
                opm_power_dBm  = np.full((n_pts, n_opm), np.nan)
                opm_spectrum_W = np.full((n_pts, n_opm, n_wl), np.nan)
                if not hdf5_ready:
                    _init_hdf5(wl_m)
                    hdf5_ready = True

            # Store in memory
            T_port1_dB    [s_idx, :]    = t1
            T_port2_dB    [s_idx, :]    = t2
            opm_power_dBm [s_idx, :]    = opm_p
            opm_spectrum_W[s_idx, :, :] = opm_s
            computed      [s_idx]       = True

            # Write to HDF5 immediately (fault-safe)
            with h5py.File(HDF5_PATH, "r+") as hf:
                hf["results/T_port1_dB"]    [s_idx, :]    = t1
                hf["results/T_port2_dB"]    [s_idx, :]    = t2
                hf["results/opm_power_dBm"] [s_idx, :]    = opm_p
                hf["results/opm_spectrum_W"][s_idx, :, :] = opm_s
                hf["flags/computed"]        [s_idx]       = True
                hf.flush()

            runs_done += 1
            if runs_done % 5 == 0 or runs_done == runs_total:
                el   = time.time() - t_start
                rate = runs_done / el if el > 0 else 1e-9
                eta  = (runs_total - runs_done) / rate
                log.info(
                    f"  [{runs_done:3d}/{runs_total}]  "
                    f"neff={neff_val:.6f}  ng={ng_val:.6f}  │  "
                    f"{rate:.2f} sim/s  │  ETA {eta:5.0f} s"
                )

        if hdf5_ready:
            with h5py.File(HDF5_PATH, "r+") as hf:
                hf["metadata"].attrs["timestamp_end"]  = datetime.now().isoformat()
                hf["metadata"].attrs["runs_completed"] = int(computed.sum())

    finally:
        try:
            ic.close()
        except Exception:
            pass
        log.info("INTERCONNECT session closed.")

    el = time.time() - t_start
    log.info(
        f"Sweep done │ {runs_done} new runs │ "
        f"total = {el:.1f} s │ avg = {el / max(runs_done, 1):.2f} s/sim"
    )
    return _pack_results(wavelengths_m, T_port1_dB, T_port2_dB,
                         opm_power_dBm, opm_spectrum_W, computed)


def _pack_results(wl, t1, t2, opm_p, opm_s, comp) -> Dict[str, Any]:
    return dict(
        neff_sweep     = SWEEP_NEFF,
        ng_sweep       = SWEEP_NG,
        wavelengths_m  = wl,
        T_port1_dB     = t1,
        T_port2_dB     = t2,
        opm_power_dBm  = opm_p,
        opm_spectrum_W = opm_s,
        computed       = comp,
    )


# ─────────────────────────────────────────────────────────────────────────────
# Execute Cell 3
# ─────────────────────────────────────────────────────────────────────────────
sweep_results = run_interconnect_sweep(hide_gui=False)

neff_sweep     = sweep_results["neff_sweep"]
ng_sweep       = sweep_results["ng_sweep"]
wavelengths_m  = sweep_results["wavelengths_m"]
T_port1_dB     = sweep_results["T_port1_dB"]
T_port2_dB     = sweep_results["T_port2_dB"]
opm_power_dBm  = sweep_results["opm_power_dBm"]
opm_spectrum_W = sweep_results["opm_spectrum_W"]
computed       = sweep_results["computed"]
wavelengths_nm = wavelengths_m * 1e9 if wavelengths_m is not None else None

print(f"\n  Sweep complete — {computed.sum()} / {len(computed)} pts computed")
if wavelengths_m is not None:
    print(f"  T_port1_dB shape    : {T_port1_dB.shape}")
    print(f"  opm_power_dBm shape : {opm_power_dBm.shape}")
print(f"  HDF5                : {HDF5_PATH}")

# ═════════════════════════════════════════════════════════════════════════════
#  END CELL 3
# ═════════════════════════════════════════════════════════════════════════════


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CELL 4 — Post-processing · Visualisation · Cache-aware re-plotting        ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

plt.rcParams.update({
    "font.family"    : "serif",
    "font.serif"     : ["DejaVu Serif", "Georgia", "Times New Roman"],
    "font.size"      : 11,
    "axes.labelsize" : 12,
    "axes.titlesize" : 13,
    "legend.fontsize": 9,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.linewidth" : 0.8,
    "axes.grid"      : True,
    "grid.alpha"     : 0.3,
    "grid.linewidth" : 0.5,
    "lines.linewidth": 1.4,
    "figure.dpi"     : 120,
    "savefig.dpi"    : 300,
    "savefig.bbox"   : "tight",
})


def load_results(path: Path = HDF5_PATH) -> Dict[str, Any]:
    """Load sweep results from HDF5 (use when Cell 3 was not run this session)."""
    if not path.exists():
        raise FileNotFoundError(f"HDF5 not found: {path}")
    with h5py.File(path, "r") as f:
        return dict(
            neff_sweep     = f["metadata/neff_sweep"][:],
            ng_sweep       = f["metadata/ng_sweep"][:],
            wavelengths_m  = f["metadata/wavelengths_m"][:],
            T_port1_dB     = f["results/T_port1_dB"][:],
            T_port2_dB     = f["results/T_port2_dB"][:],
            opm_power_dBm  = f["results/opm_power_dBm"][:],
            opm_spectrum_W = f["results/opm_spectrum_W"][:],
            computed       = f["flags/computed"][:],
        )


def get_results(path: Path = HDF5_PATH) -> Dict[str, Any]:
    """
    Return sweep_results from memory if valid, else load from HDF5.
    Raises RuntimeError with actionable instructions if neither is available.
    """
    try:
        r = sweep_results
        if r.get("wavelengths_m") is not None:
            return r
        log.warning("sweep_results in memory has wavelengths_m=None — trying HDF5 …")
    except NameError:
        pass

    if path.exists():
        log.info("Loading results from HDF5 …")
        r = load_results(path)
        if r.get("wavelengths_m") is not None:
            return r
        log.warning("HDF5 exists but wavelengths_m is empty (incomplete sweep).")

    sep = "=" * 65
    raise RuntimeError(
        f"\n{sep}\n"
        f"  No results available to plot.\n\n"
        f"  ► Run Cell 3 first to execute the sweep.\n"
        f"  ► Cell 4 can only plot data that exists in memory\n"
        f"    or is saved in the HDF5 file:\n"
        f"    {path}\n"
        f"{sep}"
    )


def _valid_mask(results: Dict) -> np.ndarray:
    return results["computed"].astype(bool)


# ─────────────────────────────────────────────────────────────────────────────
# Plot 1 — ONA Transmission spectra (colour-mapped by neff)
# ─────────────────────────────────────────────────────────────────────────────
def plot_transmittance_sweep(
    results   : Optional[Dict] = None,
    port      : int            = 1,
    n_curves  : int            = 8,
    figsize   : tuple          = (10, 5),
    cmap_name : str            = "plasma",
    save      : bool           = True,
) -> plt.Figure:
    """
    Overlay ONA transmission spectra for n_curves evenly-spaced sweep points.
    Lines are colour-mapped to Ring 1 neff.
    port=1 → RING_1 through (ONA input 1)
    port=2 → RING_14 through (ONA input 2)
    """
    if results is None:
        results = get_results()

    neff_arr = results["neff_sweep"]
    wl_nm    = results["wavelengths_m"] * 1e9
    T_data   = results["T_port1_dB"] if port == 1 else results["T_port2_dB"]
    mask     = _valid_mask(results)

    valid_idx = np.where(mask)[0]
    n_sel     = min(n_curves, len(valid_idx))
    sel_idx   = valid_idx[
        np.round(np.linspace(0, len(valid_idx) - 1, n_sel)).astype(int)
    ]

    cmap = plt.get_cmap(cmap_name)
    norm = Normalize(vmin=neff_arr[sel_idx].min(), vmax=neff_arr[sel_idx].max())
    sm   = ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])

    fig, ax = plt.subplots(figsize=figsize)
    for idx in sel_idx:
        ax.plot(wl_nm, T_data[idx], color=cmap(norm(neff_arr[idx])),
                lw=1.2, alpha=0.88)

    cbar = fig.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label("Ring 1 — $n_{eff}$ (TE)", fontsize=11)
    port_label = ("Ring 1 through  [ONA port 1]" if port == 1
                  else "Ring 14 through  [ONA port 2]")
    ax.set_xlabel("Wavelength  (nm)")
    ax.set_ylabel("Transmission  (dB)")
    ax.set_title(
        f"ONA Transmission — {port_label}  [Unidirectional rings]\n"
        f"({n_sel} of {int(mask.sum())} sweep pts,  colour = $n_{{eff,1}}$)"
    )
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    fig.tight_layout()

    if save:
        stem = f"transmittance_port{port}_{n_sel}curves"
        fig.savefig(FIGURES_DIR / f"{stem}.png")
        fig.savefig(FIGURES_DIR / f"{stem}.pdf")
        log.info(f"Saved → {stem}.png/pdf")
    return fig


# ─────────────────────────────────────────────────────────────────────────────
# Plot 2 — OPM mean power vs neff  (all 13 OPMs)
# ─────────────────────────────────────────────────────────────────────────────
def plot_opm_power_vs_neff(
    results   : Optional[Dict] = None,
    figsize   : tuple          = (11, 5),
    cmap_name : str            = "tab20",
    save      : bool           = True,
) -> plt.Figure:
    """Mean OPM power [dBm] vs Ring 1 neff — all 13 OPMs on one axes."""
    if results is None:
        results = get_results()

    neff_arr  = results["neff_sweep"]
    opm_power = results["opm_power_dBm"]
    mask      = _valid_mask(results)
    neff_v    = neff_arr[mask]
    opm_v     = opm_power[mask, :]
    n_opm     = opm_v.shape[1]
    cmap      = plt.get_cmap(cmap_name)
    colors    = [cmap(i / n_opm) for i in range(n_opm)]

    fig, ax = plt.subplots(figsize=figsize)
    for k in range(n_opm):
        ax.plot(neff_v, opm_v[:, k], color=colors[k], lw=1.3,
                marker="o", ms=3.5, markevery=max(1, len(neff_v) // 12),
                label=f"OPM {k+1}  (Ring {k+2} drop)")

    ax.set_xlabel("Ring 1 — $n_{eff}$ (TE)")
    ax.set_ylabel("Mean power  (dBm)")
    ax.set_title(
        "OPM Mean Drop Power  vs  Ring 1 $n_{eff}$  [Unidirectional]\n"
        "(Each line = one ring's drop port, integrated over ONA bandwidth)"
    )
    ax.legend(loc="upper right", ncol=2, framealpha=0.85, fontsize=8)
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    fig.tight_layout()

    if save:
        fig.savefig(FIGURES_DIR / "opm_power_vs_neff.png")
        fig.savefig(FIGURES_DIR / "opm_power_vs_neff.pdf")
        log.info("Saved → opm_power_vs_neff.png/pdf")
    return fig


# ─────────────────────────────────────────────────────────────────────────────
# Plot 3 — OPM spectral power heatmap  (neff × wavelength, per OPM)
# ─────────────────────────────────────────────────────────────────────────────
def plot_opm_spectrum_heatmap(
    results         : Optional[Dict]      = None,
    opm_ids         : Optional[List[int]] = None,
    figsize_per_row : tuple               = (10, 2.6),
    cmap_name       : str                 = "inferno",
    vmin_dBm        : Optional[float]     = None,
    vmax_dBm        : Optional[float]     = None,
    save            : bool                = True,
) -> plt.Figure:
    """2D heatmap per OPM: X=wavelength, Y=Ring 1 neff, colour=power [dBm]."""
    if results is None:
        results = get_results()

    neff_arr = results["neff_sweep"]
    wl_nm    = results["wavelengths_m"] * 1e9
    opm_spec = results["opm_spectrum_W"]
    mask     = _valid_mask(results)
    neff_v   = neff_arr[mask]
    spec_v   = opm_spec[mask, :, :]
    n_opm    = spec_v.shape[1]

    if opm_ids is None:
        opm_ids = list(range(1, n_opm + 1))

    spec_dBm = 10.0 * np.log10(np.where(spec_v > 0, spec_v, 1e-30) * 1e3)
    _vmin = vmin_dBm if vmin_dBm is not None else np.nanpercentile(spec_dBm, 2)
    _vmax = vmax_dBm if vmax_dBm is not None else np.nanpercentile(spec_dBm, 98)

    n_rows = len(opm_ids)
    fig, axes = plt.subplots(
        n_rows, 1,
        figsize=(figsize_per_row[0], figsize_per_row[1] * n_rows),
        sharex=True, sharey=True,
    )
    if n_rows == 1:
        axes = [axes]

    for ax_i, opm_id in enumerate(opm_ids):
        k   = opm_id - 1
        img = axes[ax_i].pcolormesh(
            wl_nm, neff_v, spec_dBm[:, k, :],
            cmap=cmap_name, vmin=_vmin, vmax=_vmax, shading="auto",
        )
        axes[ax_i].set_ylabel(f"OPM {opm_id}\nRing {k+2}\n$n_{{eff,1}}$", fontsize=9)
        cbar = fig.colorbar(img, ax=axes[ax_i], pad=0.01)
        cbar.set_label("dBm", fontsize=8)
        cbar.ax.tick_params(labelsize=8)

    axes[-1].set_xlabel("Wavelength  (nm)")
    fig.suptitle(
        "OPM Spectral Power  vs  Ring 1 $n_{eff}$  [Unidirectional]\n"
        "(Bright band = resonance; tracks shift as neff changes)",
        fontsize=12, y=1.01,
    )
    fig.tight_layout()

    if save:
        tag   = "_".join(str(o) for o in opm_ids)
        fname = f"opm_spectrum_heatmap_{tag}"
        fig.savefig(FIGURES_DIR / f"{fname}.png")
        fig.savefig(FIGURES_DIR / f"{fname}.pdf")
        log.info(f"Saved → {fname}.png/pdf")
    return fig


# ─────────────────────────────────────────────────────────────────────────────
# Plot 4 — Resonance wavelength tracking vs neff
# ─────────────────────────────────────────────────────────────────────────────
def plot_resonance_tracking(
    results : Optional[Dict] = None,
    port    : int            = 1,
    figsize : tuple          = (7, 4.5),
    save    : bool           = True,
) -> plt.Figure:
    """
    Find deepest transmission dip at each sweep point, plot vs neff.
    Linear fit gives ∂λ/∂neff sensitivity in nm/RIU.
    """
    if results is None:
        results = get_results()

    neff_arr = results["neff_sweep"]
    wl_nm    = results["wavelengths_m"] * 1e9
    T_data   = results["T_port1_dB"] if port == 1 else results["T_port2_dB"]
    mask     = _valid_mask(results)
    neff_v   = neff_arr[mask]
    T_v      = T_data[mask, :]
    dip_idx  = np.argmin(T_v, axis=1)
    lam_dip  = wl_nm[dip_idx]
    coeffs   = np.polyfit(neff_v, lam_dip, 1)
    poly     = np.poly1d(coeffs)
    sens     = coeffs[0]

    fig, ax = plt.subplots(figsize=figsize)
    ax.scatter(neff_v, lam_dip, s=20, zorder=5, color="#2166ac",
               label="Resonance dip")
    ax.plot(neff_v, poly(neff_v), "r--", lw=1.5,
            label=f"Linear fit   ∂λ/∂n = {sens:.3f} nm / RIU")
    ax.set_xlabel("Ring 1 — $n_{eff}$ (TE)")
    ax.set_ylabel("Resonance wavelength  (nm)")
    ax.set_title(
        f"Resonance Tracking — ONA Port {port}  [Unidirectional]\n"
        f"Sensitivity: {sens:.3f} nm / RIU"
    )
    ax.legend(framealpha=0.9)
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    fig.tight_layout()

    if save:
        fname = f"resonance_tracking_port{port}"
        fig.savefig(FIGURES_DIR / f"{fname}.png")
        fig.savefig(FIGURES_DIR / f"{fname}.pdf")
        log.info(f"Saved → {fname}.png/pdf")

    log.info(f"Resonance sensitivity (port {port}): {sens:.4f} nm / RIU")
    return fig


# ─────────────────────────────────────────────────────────────────────────────
# Registries — clean hooks for future cells
# ─────────────────────────────────────────────────────────────────────────────
PLOT_REGISTRY: Dict[str, Any] = {
    "transmittance_port1"   : lambda **kw: plot_transmittance_sweep(port=1, **kw),
    "transmittance_port2"   : lambda **kw: plot_transmittance_sweep(port=2, **kw),
    "opm_power_vs_neff"     : plot_opm_power_vs_neff,
    "opm_spectrum_heatmap"  : plot_opm_spectrum_heatmap,
    "resonance_tracking_p1" : lambda **kw: plot_resonance_tracking(port=1, **kw),
    "resonance_tracking_p2" : lambda **kw: plot_resonance_tracking(port=2, **kw),
}

DEPENDENCY_REGISTRY: Dict[str, Any] = {
    # Element naming
    "ring_name"                : ring_name,
    "opm_name"                 : opm_name,
    "ONA_NAME"                 : ONA_NAME,
    # Scripting primitives
    "_eval"                    : _eval,
    "_try_eval"                : _try_eval,
    "_fmt"                     : _fmt,
    "_add_element"             : _add_element,
    "_set_first_valid"         : _set_first_valid,
    "_RESOLVED"                : _RESOLVED,
    # Circuit + sweep
    "_apply_ring_params"       : _apply_ring_params,
    "_update_ring1_neff_ng"    : _update_ring1_neff_ng,
    "_build_circuit"           : _build_circuit,
    "_extract_results"         : _extract_results,
    "_init_hdf5"               : _init_hdf5,
    "run_interconnect_sweep"   : run_interconnect_sweep,
    # Data I/O
    "load_results"             : load_results,
    "get_results"              : get_results,
    # Plots
    "PLOT_REGISTRY"            : PLOT_REGISTRY,
    "plot_transmittance_sweep" : plot_transmittance_sweep,
    "plot_opm_power_vs_neff"   : plot_opm_power_vs_neff,
    "plot_opm_spectrum_heatmap": plot_opm_spectrum_heatmap,
    "plot_resonance_tracking"  : plot_resonance_tracking,
    # Paths / constants
    "DATA_DIR"                 : DATA_DIR,
    "FIGURES_DIR"              : FIGURES_DIR,
    "HDF5_PATH"                : HDF5_PATH,
    "VERSION_NAME"             : VERSION_NAME,
    "N_RINGS"                  : N_RINGS,
    "SPEED_OF_LIGHT"           : SPEED_OF_LIGHT,
    # Candidate / property dicts — extend in future cells if needed
    "RING_LIBRARY_CANDIDATES"  : RING_LIBRARY_CANDIDATES,
    "ONA_LIBRARY_CANDIDATES"   : ONA_LIBRARY_CANDIDATES,
    "OPM_LIBRARY_CANDIDATES"   : OPM_LIBRARY_CANDIDATES,
    "RING_PROP"                : RING_PROP,
    "ONA_PROP"                 : ONA_PROP,
}


# ─────────────────────────────────────────────────────────────────────────────
# Execute Cell 4
# ─────────────────────────────────────────────────────────────────────────────
try:
    _res = get_results()

except RuntimeError as exc:
    print(str(exc))
    print(
        "\n  ╔═══════════════════════════════════════════════════════════╗\n"
        "  ║  Run Cell 3 first to execute the sweep.                  ║\n"
        "  ║  Once at least one point completes, re-run this cell.   ║\n"
        "  ╚═══════════════════════════════════════════════════════════╝"
    )
    raise

else:
    fig1 = plot_transmittance_sweep(_res, port=1, n_curves=8)
    fig2 = plot_transmittance_sweep(_res, port=2, n_curves=8)
    fig3 = plot_opm_power_vs_neff(_res)
    fig4 = plot_opm_spectrum_heatmap(_res, opm_ids=[1, 4, 7, 10, 13])
    fig5 = plot_resonance_tracking(_res, port=1)

    plt.show()

    print(f"\n  Figures saved to : {FIGURES_DIR}")
    print(f"  HDF5 at          : {HDF5_PATH}")
    print(f"\n  Dependency registry keys:")
    for k in DEPENDENCY_REGISTRY:
        print(f"    {k}")

# ═════════════════════════════════════════════════════════════════════════════
#  END CELL 4
# ═════════════════════════════════════════════════════════════════════════════
#
#  ── PATTERNS FOR FUTURE CELLS ──────────────────────────────────────────────
#
#  1. Re-plot only (kernel restarted, no simulation needed):
#       r = load_results(HDF5_PATH)
#       plot_opm_power_vs_neff(r)
#
#  2. Switch back to bidirectional for comparison:
#       RING_PROP["bidir"] = ["bidirectional"]
#       # then in _apply_ring_params, change the value from 0 to 1
#
#  3. Add a new property candidate if your version uses a different name:
#       RING_PROP["bidir"].append("unidirectional")
#       _RESOLVED.pop("ring_bidir", None)   # clear cache to re-probe
#
#  4. Extend the sweep to include radius variation:
#       for r_val in np.linspace(18e-6, 20e-6, 5):
#           RING_RADIUS_M[0] = r_val
#           _RESOLVED.clear()   # force re-resolution after param change
#           results_r = run_interconnect_sweep()
#
#  5. Run all plots from registry:
#       for name, fn in PLOT_REGISTRY.items():
#           fn(results=_res, save=True)
# ═════════════════════════════════════════════════════════════════════════════

21:52:39 │ INFO │ No cache found — starting fresh.
21:52:39 │ INFO │ Launching Lumerical INTERCONNECT …


lumapi imported successfully from:
  C:\Program Files\Lumerical\v202\api\python\lumapi.py

  Data directory : C:\Users\jero0\OneDrive\Escritorio\Github\GDS_py_TDY_venv\Fabrication_designs\Lumerical_scripts\data_ICNT_cascade_ring_sweep
  HDF5 output    : C:\Users\jero0\OneDrive\Escritorio\Github\GDS_py_TDY_venv\Fabrication_designs\Lumerical_scripts\data_ICNT_cascade_ring_sweep\ICNT_14Ring_Cascade_UniDir_neff_sweep_V1.h5
  Figures dir    : C:\Users\jero0\OneDrive\Escritorio\Github\GDS_py_TDY_venv\Fabrication_designs\Lumerical_scripts\data_ICNT_cascade_ring_sweep\figures
  INTERCONNECT 14-Ring Cascade — Parameter Summary  [UNIDIRECTIONAL]
   Ring     R [µm]    λ_res [nm]    neff_TE     ng_TE     κ²_in     κ²_dr     Loss
  ────────────────────────────────────────────────────────────────────
      1    19.0021     1550.0000   1.609803  2.020543  0.145778  0.143402    101.0  ← swept
      2    19.1818     1550.0000   1.633303  1.991101  0.145072  0.142672    101.0
      3    19.1934     1550

21:52:43 │ INFO │   ONA added and configured.
21:52:43 │ WARNING │   ['RING_1'] No valid property found for 'bidirectional'.
  Candidates tried: ['configuration']
  ► Check exact name in GUI (double-click → Properties). — element default will be used.
21:52:43 │ INFO │   Ring 1 added  [unidir, neff=1.609803].
21:52:43 │ WARNING │   ['RING_2'] No valid property found for 'bidirectional'.
  Candidates tried: ['configuration']
  ► Check exact name in GUI (double-click → Properties). — element default will be used.
21:52:43 │ INFO │   Ring 2 added  [unidir, neff=1.633303].
21:52:44 │ WARNING │   ['RING_3'] No valid property found for 'bidirectional'.
  Candidates tried: ['configuration']
  ► Check exact name in GUI (double-click → Properties). — element default will be used.
21:52:44 │ INFO │   Ring 3 added  [unidir, neff=1.633121].
21:52:44 │ WARNING │   ['RING_4'] No valid property found for 'bidirectional'.
  Candidates tried: ['configuration']
  ► Check exact name in GUI (double-click 


  Sweep complete — 21 / 21 pts computed
  HDF5                : C:\Users\jero0\OneDrive\Escritorio\Github\GDS_py_TDY_venv\Fabrication_designs\Lumerical_scripts\data_ICNT_cascade_ring_sweep\ICNT_14Ring_Cascade_UniDir_neff_sweep_V1.h5

  No results available to plot.

  ► Run Cell 3 first to execute the sweep.
  ► Cell 4 can only plot data that exists in memory
    or is saved in the HDF5 file:
    C:\Users\jero0\OneDrive\Escritorio\Github\GDS_py_TDY_venv\Fabrication_designs\Lumerical_scripts\data_ICNT_cascade_ring_sweep\ICNT_14Ring_Cascade_UniDir_neff_sweep_V1.h5

  ╔═══════════════════════════════════════════════════════════╗
  ║  Run Cell 3 first to execute the sweep.                  ║
  ║  Once at least one point completes, re-run this cell.   ║
  ╚═══════════════════════════════════════════════════════════╝


RuntimeError: 
=================================================================
  No results available to plot.

  ► Run Cell 3 first to execute the sweep.
  ► Cell 4 can only plot data that exists in memory
    or is saved in the HDF5 file:
    C:\Users\jero0\OneDrive\Escritorio\Github\GDS_py_TDY_venv\Fabrication_designs\Lumerical_scripts\data_ICNT_cascade_ring_sweep\ICNT_14Ring_Cascade_UniDir_neff_sweep_V1.h5
=================================================================